### CreateSources (post-cutover, oxjob #548)

The sources registry lives in the **openalex-sources** Heroku Postgres and is maintained by
that app's feed / merge / curation jobs. This notebook no longer *builds* sources — it
materializes a daily, run-consistent snapshot of the federated registry
(`openalex_sources.public.sources`) in the legacy 40-column shape, so downstream consumers
(CreateSourcesApi, CreateLocationsWithSources, CreateWorksBase, CreateInstitutionsApi) run
unchanged. Runs as a plain SQL-warehouse task (`source: GIT`) — the DLT pipeline is retired.

Legacy-shape contract: `webpage` = `homepage_url`; JSONB columns parsed to typed arrays;
date columns cast to string. `issns` = registry array **verbatim** — NULL when the source
has no ISSNs (uniform convention across the registry and this snapshot; the mixed NULL/[]
compat shim via `legacy_empty_issns_sources` was retired 2026-07-09, verified churn-free
2026-07-10 — the works content hash in CreateWorksEnriched is []-blind). `issn_l` carries
the registry name directly; the inverted legacy alias `issn` was dropped 2026-07-10 (C9)
after CreateWorksBase + CreateSourcesApi were repointed to `issn_l`.
Merged sources are **included** as redirect rows (`merge_into_id` set);
consumers needing active-only filter `merge_into_id IS NULL`.

Also snapshots `openalex.sources.endpoint_to_source` from the registry's
`source_endpoint` link table (migrated 2026-07-08, migration 018) — read by
CreateLocationsWithSources' repo-matching tier. Same name/shape as the old static
table, so the consumer is unchanged; the registry re-points links on merges.

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.sources.sources')
-- 60-day time travel; must live in this DDL (not ALTER) because the daily
-- CREATE OR REPLACE resets any properties it doesn't restate
TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 60 days',
  'delta.deletedFileRetentionDuration' = 'interval 60 days'
)
AS SELECT
  s.id,
  endpoint_id,
  display_name,
  issn_l,
  publisher,
  homepage_url AS webpage,
  is_oa,
  type,
  from_json(apc_prices, 'array<struct<price:int,currency:string>>') AS apc_prices,
  is_society_journal,
  from_json(societies, 'array<struct<url:string,organization:string>>') AS societies,
  apc_usd,
  fatcat_id,
  wikidata_id,
  crossref_id,
  country,
  country_code,
  from_json(alternate_titles, 'array<string>') AS alternate_titles,
  publisher_id,
  institution_id,
  is_core,
  merge_into_id,
  merge_into_date,
  CAST(updated_date AS string) AS updated_date,
  CAST(created_date AS string) AS created_date,
  display_name_before_override,
  override_timestamp,
  datacite_id,
  -- verbatim registry value: NULL when the source has no ISSNs (the registry
  -- derives issns via array_agg, which never yields []). The works hash is
  -- []-blind (CreateWorksEnriched), so the legacy-[] cohort converges to NULL
  -- without churn (oxjob #548; compat table legacy_empty_issns_sources retired)
  s.issns,
  is_in_doaj,
  is_in_doaj_start_year,
  doaj_license,
  is_in_scielo,
  is_ojs,
  is_oa_high_oa_rate,
  high_oa_rate_start_year,
  is_fully_open_in_jstage,
  sample_pmh_record,
  COALESCE(from_json(datacite_ids, 'array<string>'), array()) AS datacite_ids,
  is_preprint_repository
FROM openalex_sources.public.sources s

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.sources.endpoint_to_source')
-- 60-day time travel; restated here because CREATE OR REPLACE resets properties
TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 60 days',
  'delta.deletedFileRetentionDuration' = 'interval 60 days'
)
AS SELECT
  endpoint_id,
  source_id
FROM openalex_sources.public.source_endpoint

In [ ]:
SELECT
  COUNT(*) AS total,
  COUNT_IF(merge_into_id IS NULL) AS active,
  COUNT_IF(is_in_doaj) AS in_doaj,
  MAX(id) AS max_id
FROM identifier('openalex' || :env_suffix || '.sources.sources')